# Exercise 1 - PydanticAI, FastAPI and LLM judge

2. A pedagogical programming bot


Create a chatbot that is specialized in helping out with programming tasks. It should answer shortly, so not spoonfeeding with long answers, and it should be socratic, that is always ask follow up questions rather than giving the answer.

Create LLM judges to evaluate how well the bot follows its guidelines and a judge to detect user frustration. Are there any more LLM judges that could be suitable here?

## Bygg chatboten - Imports, create agent 

In [16]:
from pydantic_ai import Agent
from dotenv import load_dotenv
from constants import MODEL_LARGE
import mlflow

load_dotenv()

system_prompt = """
You are a programming tutor chatbot.


Rules:
- Help with programming tasks only.
- Keep answers short and focused.
- Do not spoonfeed full solutions unless the user explicit asks.
- Prefer hints, debugging direction, and small next steps.
- Be Socratic: ask at least one useful follow-up question in most replies.
-If the user seems frustrated, acknowledge it briefly and reduce pressure.
-When the user provides code, prioritize identigying the smallest likely issue first.
- If you are unsure, say so briefly instead of guessing.

"""

code_helper_agent = Agent(
    MODEL_LARGE,
    system_prompt=system_prompt,
)

## Eval dataset - Gör en liten eval-dataset

Du behöver testfall där du kan avgöra om boten följer reglerna.

In [17]:
import pandas as pd

eval_data = pd.DataFrame([
    {
        "inputs": {"user_message": "Why do I get KeyError in pandas when I access a column?"},
        "expectations": {
            "expected_facts": [
                "The answer should mention that the column name may not match exactly.",
                "The answer should suggest checking df.columns.",
                "The answer should ask a follow-up question."
            ]
        },
    },
    {
        "inputs": {"user_message": "Write the full binary search solution for me."},
        "expectations": {
            "expected_facts": [
                "The answer should avoid immediately giving the full solution by default.",
                "The answer should guide the user with a smaller next step or hint.",
                "The answer should ask a follow-up question."
            ]
        },
    },
    {
        "inputs": {"user_message": "I've tried this five times and nothing works, this is so annoying."},
        "expectations": {
            "expected_facts": [
                "The answer should briefly acknowledge frustration.",
                "The answer should stay calm and not overwhelm the user.",
                "The answer should ask for the most useful next detail, such as error message, traceback, or code."
            ]
        },
    },
    {
        "inputs": {"user_message": "Can you help me plan my vacation in Spain?"},
        "expectations": {
            "expected_facts": [
                "The answer should recognize that this is outside programming scope.",
                "The answer should avoid pretending it is a programming task."
            ]
        },
    },
    {
        "inputs": {"user_message": "My FastAPI app returns 500. Where should I start debugging?"},
        "expectations": {
            "expected_facts": [
                "The answer should suggest a small debugging step.",
                "The answer should remain short.",
                "The answer should ask for traceback, logs, or code."
            ]
        },
    },
    {
        "inputs": {"user_message": "Here is my loop: while x != 10: print(x). Why does it never stop?"},
        "expectations": {
            "expected_facts": [
                "The answer should point out that x is not being updated.",
                "The answer should not dump a long lecture.",
                "The answer should ask a follow-up question."
            ]
        },
    },
])

## Run bot on all test data

In [19]:
async def run_bot(user_message: str) -> str:
    result = await code_helper_agent.run(user_message)
    return str(result.output)

outputs = []

for row in eval_data.to_dict(orient="records"):
    user_message = row["inputs"]["user_message"]
    bot_output = await run_bot(user_message)
    outputs.append(bot_output)

eval_data = pd.DataFrame(eval_data)
eval_data["outputs"] = outputs
eval_data[["inputs", "outputs", "expectations"]]

Traceback (most recent call last):
  File "c:\Users\tasmi\MLOPS\llmops_sadia_awan_mlo25\.venv\Lib\site-packages\pydantic_ai\models\openai.py", line 136, in _map_api_errors
    yield
  File "c:\Users\tasmi\MLOPS\llmops_sadia_awan_mlo25\.venv\Lib\site-packages\pydantic_ai\models\openai.py", line 802, in _completions_create
    return await self.client.chat.completions.create(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<29 lines>...
    )
    ^
  File "c:\Users\tasmi\MLOPS\llmops_sadia_awan_mlo25\.venv\Lib\site-packages\openai\resources\chat\completions\completions.py", line 2714, in create
    return await self._post(
           ^^^^^^^^^^^^^^^^^
    ...<49 lines>...
    )
    ^
  File "c:\Users\tasmi\MLOPS\llmops_sadia_awan_mlo25\.venv\Lib\site-packages\openai\_base_client.py", line 1884, in post
    return await self.request(cast_to, opts, stream=stream, stream_cls=stream_cls)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c

## Code scorer: shortness and follow up question?

In [20]:
from mlflow.entities import Feedback
from mlflow.genai import scorer

@scorer
def brevity_check(outputs, str) -> Feedback:
    word_count = len(str(outputs).split())
    passed = word_count <= 120
    rationale = f"word_count{word_count}"
    return Feedback(value=passed, rationale=rationale)

@scorer
def followup_question_check(outputs, str) -> Feedback:
    text = str(outputs)
    passed = "?" in text
    rationale = "Contains a follow-up question." if passed else "No question mark found"
    return Feedback(value=passed, rationale=rationale)

## Guideline Judge

In [21]:
from mlflow.genai.scorers import Correctness, Guidelines
from mlflow.genai.judges import make_judge

JUDGE_MODEL ="openrouter:/nvidia/nemotron-3-nano-30b-a3b:free"

bot_guidelines = Guidelines(
    name="bot_guideline_adherence",
    guidelines=[
        "The response must stay focused on programming help.",
        "The response must be short and concise.",
        "The response must not spoonfeed a full solution unless the user explicitly asked for a full solution.",
        "The response must ask a useful follow-up question."
    ],
    model=JUDGE_MODEL,
)

## Frustration Judge

In [22]:
from typing import Literal

frustration_detection_judge = make_judge(
    name="frustration_detection",
    instructions="""
You are judging whether the user message shows frustration.

Read {{ inputs }} only.

Return:
"frustrated" if the user sounds annoyed, upset, overwhelmed, or emotionally negative
"not_frustrated" otherwise

Be conservative:
confusion alone is not enough
clear phrases like "nothing works", "this is annoying", "I've tried everything", "I'm stuck" indicate frustration
""",
    feedback_value_type=Literal["frustrated", "not_frustrated"],
    model=JUDGE_MODEL,
)

## Socratic Judge

In [23]:
socratic_judge = make_judge(
    name="socratic_quality",
    instructions="""
Evaluate whether the assistant response is genuinely Socratic.

Use {{ inputs }} and {{ outputs }}.

Return "yes" only if:
the answer asks a relevant follow-up question
the question helps the user think or debug further
the answer does not mainly dump the final solution immediately

Return "no" otherwise.
""",
    feedback_value_type=Literal["yes", "no"],
    model=JUDGE_MODEL,
)

## Anti spoonfeeding Judge

In [24]:
anti_spoonfeeding_judge = make_judge(
    name="anti_spoonfeeding",
    instructions="""
Evaluate whether the assistant avoids spoonfeeding.

Use {{ inputs }} and {{ outputs }}.

Return "pass" if:
the response gives a hint, next step, or guiding question
and it does not immediately provide a full finished solution unless explicitly requested

Return "fail" otherwise.
""",
    feedback_value_type=Literal["pass", "fail"],
    model=JUDGE_MODEL,
)

## Built in Correctness Judge

In [25]:
correctness_judge = Correctness(model=JUDGE_MODEL)

## Run eval

In [26]:
with mlflow.start_run(run_name="programming-bot-evaluation"):
    mlflow.log_param("app_model", MODEL_LARGE)
    mlflow.log_param("judge_model", JUDGE_MODEL)

    results = mlflow.genai.evaluate(
        data=eval_data,
        scorers=[
            correctness_judge,
            bot_guidelines,
            frustration_detection_judge,
            socratic_judge,
            anti_spoonfeeding_judge,
            brevity_check,
            followup_question_check,
        ],
    )

2026/04/17 16:34:50 INFO mlflow.genai.scorers.validation: The input data is missing following columns that are required by the specified scorers. The results will be null for those scorers.
 - `outputs` column is required by [correctness, bot_guideline_adherence].
Evaluating:   0%|          | 0/6 [Elapsed: 00:00, Remaining: ?]2026/04/17 16:34:50 INFO mlflow.genai.judges.instructions_judge: Could not extract outputs from the trace root span. Falling back to trace-based judge mode.
2026/04/17 16:34:50 INFO mlflow.genai.judges.instructions_judge: Could not extract outputs from the trace root span. Falling back to trace-based judge mode.
2026/04/17 16:34:50 INFO mlflow.genai.judges.instructions_judge: Could not extract outputs from the trace root span. Falling back to trace-based judge mode.
2026/04/17 16:34:50 INFO mlflow.genai.judges.instructions_judge: Could not extract outputs from the trace root span. Falling back to trace-based judge mode.
2026/04/17 16:34:51 INFO mlflow.genai.judges


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: programming-bot-evaluation
  Run ID: c86813e7f0034435b86f91aa179cafb2

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



## Show result

In [27]:
results.result_df

,trace_id,brevity_check/value,followup_question_check/value,frustration_detection/value,bot_guideline_adherence/value,correctness/value,anti_spoonfeeding/value,socratic_quality/value,expected_facts/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-4c62c2f73a5d85bff2de84440d1d731b,None,None,None,None,None,None,None,None,"{""info"": {""trace_id"": ""tr-4c62c2f73a5d85bff2de...",None,OK,1776436490422,1,{'user_message': 'Why do I get KeyError in pan...,None,"{'mlflow.user': 'tasmi', 'mlflow.source.name':...",{'mlflow.artifactLocation': 'file:c:/Users/tas...,"[{'trace_id': 'TGLC9zpdhb/y3oREDR1zGw==', 'spa...",[{'assessment_id': 'a-6a38a1358efb46c6a5737298...
1,tr-0bc43f03f5517d6e145e7163b0a18516,None,None,None,None,None,None,None,None,"{""info"": {""trace_id"": ""tr-0bc43f03f5517d6e145e...",None,OK,1776436490423,2,{'user_message': 'Write the full binary search...,None,"{'mlflow.user': 'tasmi', 'mlflow.source.name':...",{'mlflow.artifactLocation': 'file:c:/Users/tas...,"[{'trace_id': 'C8Q/A/VRfW4UXnFjsKGFFg==', 'spa...",[{'assessment_id': 'a-1d74c7a442554333b1092aa2...
2,tr-2728372598a394bc12a11a40e7e089b1,None,None,None,None,None,None,None,None,"{""info"": {""trace_id"": ""tr-2728372598a394bc12a1...",None,OK,1776436490425,0,{'user_message': 'I've tried this five times a...,None,"{'mlflow.user': 'tasmi', 'mlflow.source.name':...",{'mlflow.artifactLocation': 'file:c:/Users/tas...,"[{'trace_id': 'Jyg3JZijlLwSoRpA5+CJsQ==', 'spa...",[{'assessment_id': 'a-a94cb22141c04c8391956b6c...
3,tr-358a2970337a1d2b65bfc0fcec280bb1,None,None,None,None,None,None,None,None,"{""info"": {""trace_id"": ""tr-358a2970337a1d2b65bf...",None,OK,1776436490427,0,{'user_message': 'Can you help me plan my vaca...,None,"{'mlflow.user': 'tasmi', 'mlflow.source.name':...",{'mlflow.artifactLocation': 'file:c:/Users/tas...,"[{'trace_id': 'NYopcDN6HStlv8D87CgLsQ==', 'spa...",[{'assessment_id': 'a-7acde77b2d134ba9a630fa26...
4,tr-efd33254803ac7e6fa08add6a7ee7be7,None,None,None,None,None,None,None,None,"{""info"": {""trace_id"": ""tr-efd33254803ac7e6fa08...",None,OK,1776436490429,0,{'user_message': 'My FastAPI app returns 500. ...,None,"{'mlflow.user': 'tasmi', 'mlflow.source.name':...",{'mlflow.artifactLocation': 'file:c:/Users/tas...,"[{'trace_id': '79MyVIA6x+b6CK3Wp+575w==', 'spa...",[{'assessment_id': 'a-f19add8b04f847cc92ce3168...
5,tr-0f74392530d86db0bcb569b4ad3d40f4,None,None,None,None,None,None,None,None,"{""info"": {""trace_id"": ""tr-0f74392530d86db0bcb5...",None,OK,1776436490432,0,{'user_message': 'Here is my loop: while x != ...,None,"{'mlflow.user': 'tasmi', 'mlflow.source.name':...",{'mlflow.artifactLocation': 'file:c:/Users/tas...,"[{'trace_id': 'D3Q5JTDYbbC8tWm0rT1A9A==', 'spa...",[{'assessment_id': 'a-30e1fe5a801345288598a3d1...
